# ScholarScan — OCR Pipeline

Extracts text from handwritten/scanned student answer sheets.

## Pipeline
```
PDF / Image
    ↓
[pdfplumber] ← digital pages (text-layer PDFs)
    ↓ (scanned / no text layer)
[OCR Cascade]
  1. Mistral OCR  → confidence threshold 0.70 → short-circuit
  2. Google Vision → confidence threshold 0.80 → short-circuit
  3. Tesseract     → last-resort offline fallback
    ↓
raw text string  →  downstream NLP
```

**Image preprocessing** (shared across Tesseract + Vision):
- Grayscale conversion
- Gaussian blur (denoise)
- Adaptive threshold (binarise)
- Deskew (correct rotation)
- Morphological open (remove noise blobs)

In [ ]:
# Install system deps (Tesseract + poppler for pdf2image)
!apt-get install -qq tesseract-ocr poppler-utils 2>/dev/null || echo 'apt-get not available (non-Colab env)'

# Python packages
!pip install -q pytesseract pdfplumber pdf2image opencv-python-headless pillow mistralai reportlab matplotlib numpy

In [ ]:
import os, base64, logging, shutil, tempfile, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Callable, List

import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger('scholarscan.ocr')
print('✓ Imports OK')

## 1. Base Types

`OcrPage` — single page result (index + markdown text + confidence).  
`OcrResult` — full extraction result from any provider.

In [ ]:
# Source: backend/adapters/ocr/base.py

@dataclass(frozen=True)
class OcrPage:
    index: int
    markdown: str
    confidence: float | None = None


@dataclass
class OcrResult:
    text: str
    confidence: float   # 0.0–1.0 (heuristic for providers without native scores)
    provider: str
    page_data: tuple = ()
    metadata: dict = field(default_factory=dict)

    @property
    def page_count(self) -> int:
        return max(1, len(self.page_data))

    def summary(self) -> str:
        return (
            f"Provider  : {self.provider}\n"
            f"Confidence: {self.confidence:.2f}\n"
            f"Pages     : {self.page_count}\n"
            f"Characters: {len(self.text)}\n"
            f"Preview   : {self.text[:200]}..."
        )

print('✓ OcrPage, OcrResult defined')

## 2. Image Preprocessing

Shared pipeline before passing to any OCR engine.

In [ ]:
# Source: backend/adapters/ocr/base.py — preprocessing helpers

def load_image_gray(image_path: str) -> np.ndarray:
    """Load image and convert to grayscale."""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot decode image: {image_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def _deskew(image: np.ndarray) -> np.ndarray:
    """Correct small rotational skew caused by scanning."""
    ys, xs = np.where(image == 0)
    coords = np.column_stack((xs, ys))
    if coords.size == 0:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    h, w = image.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def preprocess_image(image: np.ndarray) -> np.ndarray:
    """Full preprocessing: blur → binarise → deskew → morphological open."""
    blurred = cv2.GaussianBlur(image, (3, 3), 0)
    binary = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    deskewed = _deskew(binary)
    kernel = np.ones((1, 1), np.uint8)
    return cv2.morphologyEx(deskewed, cv2.MORPH_OPEN, kernel)


print('✓ Preprocessing functions defined')

In [ ]:
# Create a synthetic test image (simulates a student answer sheet)

def create_test_image(lines: list[str], path: str = "/tmp/test_answer.png",
                      width: int = 900, line_height: int = 55) -> str:
    height = len(lines) * line_height + 80
    img = Image.new("RGB", (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    # Faint ruled lines
    for y in range(80, height, line_height):
        draw.line([(30, y), (width - 30, y)], fill=(200, 210, 220), width=1)
    # Text
    y = 35
    for line in lines:
        draw.text((45, y), line, fill=(20, 20, 80))
        y += line_height
    img.save(path)
    return path


SAMPLE_LINES = [
    "Q1. Define machine learning and its main categories.",
    "",
    "Ans: Machine learning is a branch of artificial intelligence",
    "that enables systems to learn from data and improve performance",
    "without explicit programming. Main categories are:",
    "  1) Supervised learning   2) Unsupervised learning",
    "  3) Reinforcement learning",
]

img_path = create_test_image(SAMPLE_LINES)
print(f'Test image saved: {img_path}')

# Visualise raw vs preprocessed
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
orig = Image.open(img_path)
axes[0].imshow(orig)
axes[0].set_title('Original image', fontsize=13, fontweight='bold')
axes[0].axis('off')

gray = load_image_gray(img_path)
processed = preprocess_image(gray)
axes[1].imshow(processed, cmap='gray')
axes[1].set_title('After preprocessing\n(grayscale → threshold → deskew)', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('OCR Image Preprocessing', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 3. Provider A: Tesseract (Offline Fallback)

No API key needed. Works fully offline. Confidence always 0.5 (Tesseract CLI doesn't expose reliable per-image confidence).  
Used as **last resort** in the cascade.

In [ ]:
# Source: backend/adapters/ocr/tesseract.py

class TesseractProvider:
    """Local Tesseract OCR — last-resort offline fallback."""

    name = "tesseract"
    confidence_threshold = 0.0  # always accept; last in cascade

    def _ocr_array(self, image: np.ndarray) -> str:
        if shutil.which("tesseract") is None:
            raise RuntimeError("Tesseract binary not found — run: apt-get install tesseract-ocr")
        import pytesseract
        pil_image = Image.fromarray(image)
        return pytesseract.image_to_string(pil_image, config="--psm 6").strip()

    def _ocr_image(self, path: str) -> str:
        gray = load_image_gray(path)
        preprocessed = preprocess_image(gray)
        return self._ocr_array(preprocessed)

    def extract(self, path: str) -> OcrResult:
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        text = self._ocr_image(path)
        confidence = 0.5 if text.strip() else 0.0
        pages = (OcrPage(index=0, markdown=text, confidence=confidence),)
        return OcrResult(
            text=text,
            confidence=confidence,
            provider=self.name,
            page_data=pages,
        )


print('✓ TesseractProvider defined')

# Run it
tess = TesseractProvider()
try:
    result = tess.extract(img_path)
    print('\n--- Tesseract OCR Result ---')
    print(result.summary())
except RuntimeError as e:
    print(f'Tesseract not available: {e}')

## 4. Provider B: Mistral OCR (Primary)

Requires `MISTRAL_API_KEY`. Returns markdown with layout awareness (tables, headings, math).  
Confidence threshold: **0.70** — cascade short-circuits here.

In [ ]:
# Source: backend/adapters/ocr/mistral_ocr.py

_MIME_BY_SUFFIX = {
    ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png",
    ".bmp": "image/bmp",  ".tif": "image/tiff", ".tiff": "image/tiff",
    ".webp": "image/webp", ".pdf": "application/pdf",
}


class MistralOcrProvider:
    """Mistral OCR primary provider — returns markdown with layout."""

    name = "mistral"
    confidence_threshold = 0.70

    def __init__(self, api_key: str, model_name: str = "mistral-ocr-latest") -> None:
        self._api_key = api_key
        self._model_name = model_name
        self._client = self._build_client()

    def _build_client(self):
        try:
            from mistralai import Mistral
            return Mistral(api_key=self._api_key)
        except Exception as exc:
            logger.warning("Mistral init failed: %s", exc)
            return None

    def _encode(self, path: str) -> tuple[str, str]:
        suffix = Path(path).suffix.lower()
        mime = _MIME_BY_SUFFIX.get(suffix, "image/png")
        with open(path, "rb") as fh:
            b64 = base64.b64encode(fh.read()).decode("ascii")
        return mime, b64

    def extract(self, path: str) -> OcrResult:
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        if self._client is None:
            raise RuntimeError("Mistral client not available — check API key")

        mime, b64 = self._encode(path)
        data_uri = f"data:{mime};base64,{b64}"
        is_pdf_file = path.lower().endswith(".pdf")
        document = (
            {"type": "document_url", "document_url": data_uri}
            if is_pdf_file else
            {"type": "image_url", "image_url": data_uri}
        )

        response = self._client.ocr.process(
            model=self._model_name,
            document=document,
            confidence_scores_granularity="page",
        )

        pages = getattr(response, "pages", []) or []
        page_texts = [(getattr(p, "markdown", "") or "").strip() for p in pages]
        combined = "\n\n".join(t for t in page_texts if t).strip()

        page_scores = [getattr(getattr(p, "confidence_scores", None), "average_page_confidence_score", None) for p in pages]
        real_scores = [s for s in page_scores if s is not None]
        confidence = sum(real_scores) / len(real_scores) if real_scores else (0.92 if combined else 0.0)

        ocr_pages = tuple(
            OcrPage(index=i, markdown=page_texts[i], confidence=page_scores[i])
            for i in range(len(pages))
        )
        return OcrResult(
            text=combined, confidence=confidence, provider=self.name,
            page_data=ocr_pages, metadata={"model": self._model_name},
        )


print('✓ MistralOcrProvider defined')

In [ ]:
# Test Mistral OCR — set your API key here or in the env
import os
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")  # or paste key directly

if MISTRAL_API_KEY:
    mistral = MistralOcrProvider(MISTRAL_API_KEY)
    try:
        result = mistral.extract(img_path)
        print('--- Mistral OCR Result ---')
        print(result.summary())
    except Exception as e:
        print(f'Mistral OCR error: {e}')
else:
    print('⚠ MISTRAL_API_KEY not set.')
    print('  Set it above or: export MISTRAL_API_KEY=your_key')
    print('  Skipping Mistral demo — cascade will fall back to Tesseract.')

## 5. OCR Cascade Orchestrator

Chain: **Mistral → Google Vision → Tesseract**

Logic:
1. Try each provider in order
2. If `result.confidence >= provider.confidence_threshold` → short-circuit (accept result)
3. If provider fails → log warning, try next
4. If all fail → raise `RuntimeError`
5. Total wall-clock cap: 120s

In [ ]:
# Source: backend/services/ocr_service.py

class OcrCascade:
    """Cascade OCR orchestrator. Tries providers in order, short-circuits on confidence."""

    def __init__(self, providers: list, total_timeout: int = 120) -> None:
        self._providers = providers
        self._total_timeout = total_timeout

    def extract_result(self, path: str) -> OcrResult:
        if not self._providers:
            raise RuntimeError("OcrCascade: no providers configured")

        deadline = time.monotonic() + self._total_timeout
        best: OcrResult | None = None
        last_error = None

        for provider in self._providers:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                logger.warning("OcrCascade: total timeout reached")
                break

            try:
                result = provider.extract(path)
                logger.info(
                    "%s → %d chars, confidence=%.2f",
                    provider.name, len(result.text), result.confidence,
                )
                if best is None or result.confidence > best.confidence:
                    best = result
                if result.confidence >= provider.confidence_threshold:
                    logger.info(
                        "Short-circuit: %s (conf %.2f >= threshold %.2f)",
                        provider.name, result.confidence, provider.confidence_threshold,
                    )
                    return result
            except Exception as exc:
                last_error = exc
                logger.warning("%s failed: %s", provider.name, exc)

        if best is not None:
            logger.info("Cascade exhausted, returning best: %s", best.provider)
            return best

        raise RuntimeError("OcrCascade: all providers failed") from last_error

    def extract(self, path: str) -> str:
        """Legacy str-returning interface."""
        return self.extract_result(path).text


print('✓ OcrCascade defined')

# Build cascade from available providers
providers = []
if MISTRAL_API_KEY:
    providers.append(MistralOcrProvider(MISTRAL_API_KEY))
providers.append(TesseractProvider())  # always available as fallback

cascade = OcrCascade(providers)
result = cascade.extract_result(img_path)
print(f'\n=== Cascade Result ===')
print(result.summary())

## 6. PDF Service

Handles mixed PDFs: digital pages use `pdfplumber` (fast, lossless), scanned pages route to the OCR cascade.

In [ ]:
# Source: backend/services/pdf_service.py
import pdfplumber

class PdfService:
    """
    Extracts text from PDFs using pdfplumber for digital content.
    Falls back to OCR for scanned pages with no extractable text.
    """

    def __init__(
        self,
        extract_text_fn: Callable,
        ocr_extract_result_fn: Optional[Callable] = None,
    ):
        self._extract_text_fn = extract_text_fn
        self._ocr_extract_result_fn = ocr_extract_result_fn

    def is_scanned_page(self, page) -> bool:
        """True when pdfplumber finds no extractable text on a page."""
        text = page.extract_text()
        return not text or not text.strip()

    def extract_text(
        self,
        pdf_path: str,
        on_progress: Optional[Callable] = None,
    ) -> str:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF not found: {pdf_path}")

        page_texts: list[str] = []
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                if self.is_scanned_page(page):
                    ocr_text = self._ocr_page(pdf_path, i)
                    page_texts.append(ocr_text)
                    mode = "ocr"
                else:
                    text = page.extract_text() or ""
                    page_texts.append(text)
                    mode = "digital"

                if on_progress:
                    on_progress(i + 1, total_pages, mode)

        combined = "\n\n".join(page_texts).strip()
        if not combined:
            raise ValueError("No text extracted. Check OCR connectivity.")
        return combined

    def _ocr_page(self, pdf_path: str, page_index: int) -> str:
        """Convert a single PDF page to PNG → OCR."""
        try:
            from pdf2image import convert_from_path
            images = convert_from_path(
                pdf_path, first_page=page_index + 1, last_page=page_index + 1, dpi=300
            )
            if not images:
                return ""
            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                images[0].save(tmp, format="PNG")
                tmp_path = tmp.name
            try:
                return self._extract_text_fn(tmp_path)
            finally:
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)
        except Exception as exc:
            logger.warning("OCR failed for page %d: %s", page_index + 1, exc)
            return ""


print('✓ PdfService defined')

In [ ]:
# Create a sample PDF with a digital text layer and test PdfService
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas

    pdf_path = "/tmp/sample_answer_sheet.pdf"
    c = canvas.Canvas(pdf_path, pagesize=A4)
    c.setFont("Helvetica", 12)
    lines = [
        "Student: Test Student",
        "Subject: Artificial Intelligence",
        "",
        "Q1. What is supervised learning?",
        "",
        "Ans: Supervised learning is a type of machine learning where",
        "the model is trained on labelled data. The algorithm learns",
        "a mapping from inputs to outputs using training examples.",
        "Common algorithms include SVM, decision trees, and neural networks.",
    ]
    y = 780
    for line in lines:
        c.drawString(60, y, line)
        y -= 22
    c.save()
    print(f'Sample PDF created: {pdf_path}')

    # Extract using PdfService (digital mode, no OCR needed)
    def progress_callback(current, total, mode):
        print(f'  Page {current}/{total} — mode: {mode}')

    pdf_svc = PdfService(extract_text_fn=cascade.extract)
    text = pdf_svc.extract_text(pdf_path, on_progress=progress_callback)
    print(f'\n=== Extracted text ===')
    print(text)

except ImportError:
    print('reportlab not installed — run: pip install reportlab')

## 7. Cascade Confidence Flow Visualisation

In [ ]:
# Visualise how confidence determines cascade flow

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# Boxes
boxes = [
    (0.5, 3.5, 'Input\n(PDF/Image)', '#4a90d9', 'white'),
    (2.5, 4.5, 'Mistral\nOCR', '#27ae60', 'white'),
    (2.5, 2.5, 'Google\nVision', '#f39c12', 'white'),
    (2.5, 0.5, 'Tesseract\n(offline)', '#e74c3c', 'white'),
    (7.5, 2.5, 'OcrResult\n(text + confidence)', '#8e44ad', 'white'),
]

for x, y, label, color, tc in boxes:
    rect = mpatches.FancyBboxPatch((x - 0.9, y - 0.55), 1.8, 1.1,
                                    boxstyle='round,pad=0.1', facecolor=color,
                                    edgecolor='white', linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', color=tc,
            fontsize=9, fontweight='bold', zorder=4)

# Arrows
arrow_kw = dict(arrowstyle='->', color='#555', lw=1.5)
ax.annotate('', xy=(1.6, 3.5), xytext=(1.6, 4.5), arrowprops=dict(arrowstyle='->', color='#4a90d9', lw=2))
ax.annotate('', xy=(1.6, 4.5), xytext=(0.5, 4.5), arrowprops=dict(arrowstyle='->', color='#4a90d9', lw=2))
ax.annotate('', xy=(1.6, 2.5), xytext=(1.6, 3.5), arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax.annotate('', xy=(1.6, 0.5), xytext=(1.6, 2.5), arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax.annotate('', xy=(2.5-0.9, 4.5), xytext=(1.6, 4.5), arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax.annotate('', xy=(2.5-0.9, 2.5), xytext=(1.6, 2.5), arrowprops=dict(arrowstyle='->', color='#f39c12', lw=2))
ax.annotate('', xy=(2.5-0.9, 0.5), xytext=(1.6, 0.5), arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
for y_from, color in [(4.5, '#27ae60'), (2.5, '#f39c12'), (0.5, '#e74c3c')]:
    ax.annotate('', xy=(6.6, 2.5), xytext=(3.4, y_from),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5,
                                connectionstyle='arc3,rad=0.15'))

# Labels
ax.text(4.8, 5.1, 'conf >= 0.70 → short-circuit', fontsize=8.5, color='#27ae60', style='italic')
ax.text(4.8, 3.7, 'conf >= 0.80 → short-circuit', fontsize=8.5, color='#f39c12', style='italic')
ax.text(4.8, 1.5, 'conf = 0.5 (always accepted)', fontsize=8.5, color='#e74c3c', style='italic')

ax.text(1.6, 5.3, 'Cascade order (priority)', ha='center', fontsize=10, fontweight='bold', color='#333')
ax.text(5, 0.1, 'Best result returned if threshold never met', ha='center', fontsize=9, color='#666', style='italic')

plt.title('ScholarScan OCR Cascade — Provider Flow', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 8. End-to-End Test

Upload your own image or PDF to test the full pipeline.

In [ ]:
# ── OPTION A: Upload file in Colab ──
try:
    from google.colab import files
    print('Running in Colab — use the upload widget:')
    uploaded = files.upload()
    if uploaded:
        uploaded_path = list(uploaded.keys())[0]
        result = cascade.extract_result(uploaded_path)
        print(f'\n=== OCR Result ===')
        print(result.summary())
except ImportError:
    pass

# ── OPTION B: Provide a local file path ──
custom_path = ""  # e.g. "/content/my_answer.jpg" or "answer.pdf"
if custom_path and os.path.exists(custom_path):
    result = cascade.extract_result(custom_path)
    print(f'\n=== OCR Result ===')
    print(result.summary())
elif not custom_path:
    # Fall back to the synthetic test image
    result = cascade.extract_result(img_path)
    print('Using synthetic test image:')
    print(result.summary())